# Create Text Embedding Models

## What is Contrastive Learning?
- Contrastive learning is a technique that aims to train an embedding model such that similar documents are closer in vector space while dissimilar documents are further apart.
- The underlying idea of contrastive learning is that the best way to learn and model similarity/dissimilarity between documents is by feeding a model examples of similar and dissimilar pairs.
- In order to accurately capture the semantic nature of a document, it often needs to be contrasted with another document for a model to learn what makes it different or similar
- This contrasting procedure is quite powerful and relates to the context in which documents are written. This high-level procedure is demonstrated in
- There are many ways we can apply contrastive learning to create text embedding models but the most well-known technique and framework is sentence-transformers.

## SBERT
 - Cross-encoder allows two sentences to be passed to the transformer network simultaneously to predict the extent to which the two sentences are similar.
 - It does so by adding a classification head to the original architecture that can output a similarity score. 
 - However, the number of computations rises quickly when you want to find the highest pair in a collection of 10,000 sentences. That would require n·(n−1)/2 = 49,995,000 inference computations and therefore generates significant overhead.
 -  Moreover, a cross-encoder generally does not generate embeddings, as shown in img6. Instead, it outputs a similarity score between the input sentences.

 - Instead, the authors of sentence-transformers approached the problem differently and searched for a method that is fast and creates embeddings that can be compared semantically. 
   - The result is an elegant alternative to the original cross-encoder architecture. 
   - Unlike a cross-encoder, in sentence-transformers the classification head is dropped, and instead mean pooling is used on the final output layer to generate an embedding. 
   - This pooling layer averages the word embeddings and gives back a fixed dimensional output vector. This ensures a fixed-size embedding.

## Crating an embedding model
### Generating Contrastive Examples
- When pretraining your embedding model, you will often see data being used from natural language inference (NLI) datasets. 
- NLI refers to the task of investigating whether, for a given premise, it entails the hypothesis (entailment), contradicts it (contradiction), or neither (neutral).

- In our Example, we'll use GLUE dataset that consistes of 9 language understanding tasks to evaluate an anlyze the model performance.
- One of these tasks is the Multi-Genre Natural Language Inference (MNLI) corpus, which is a collection of 392,702 sentence pairs annotated with entailment (contradiction, neutral, entailment). 
- We will be using a subset of the data, 50,000 annotated sentence pairs, to create a minimal example that does not need to be trained for hours on end.


In [1]:
pip install -q accelerate>=0.27.2 peft>=0.9.0 bitsandbytes>=0.43.0 transformers>=4.38.2 trl>=0.7.11 sentencepiece>=0.1.99

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -q sentence-transformers>=3.0.0 mteb>=1.1.2 datasets>=2.18.0

Note: you may need to restart the kernel to use updated packages.


### Data

In [1]:
from datasets import load_dataset
# Load MNLI dataset from GLUE
# 0=entailment, 1=neutral, 2=contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train_dataset[0]

{'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.',
 'hypothesis': 'Product and geography are what make cream skimming work. ',
 'label': 1}

### Train the model
- Now that we have our dataset with training examples, we will need to create our embedding model.
- We typically choose an existing sentence-transformers model and fine-tune that model, but in this example, we are going to train an embedding from scratch.
- This means that we will have to define two things.
  -  **First, a pretrained Transformer model that serves as embedding individual words.**
    - We will use the BERT base model (uncased) as it is a great introduction model.
    - However, many others exist that also have been evaluated using sentence-transformers.  Most notably, microsoft/mpnet-base often gives good results when used as a word embedding model: https://www.sbert.net/docs/sentence_transformer/training_overview.html#best-transformer-model
  

In [2]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("bert-base-uncased")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3618.22it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


- Next, we will need to define a loss function over which we will optimize the model. 
### Loss function


In [3]:
from sentence_transformers import losses

# Define the loss function. In soft-max loss, we will also need to explicitly set the number of labels.
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3
)

C:\Users\pc\AppData\Local\Temp\ipykernel_29608\1944009623.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses
C:\Users\pc\AppData\Local\Temp\ipykernel_29608\1944009623.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
The `sentence_embedding_dimension` argument was renamed and is now deprecated. Please use `embedding_dimension` instead.


- Before we train our model, we define an evaluator to evaluate the model’s performance during training, which also determines the best model to save.
- We can perform evaluation of the performance of our model using the Semantic Textual Similarity Benchmark (STSB).
  - It is a collection of human-labeled sentence pairs, with similarity scores between 1 and 5.

### Evaluation


In [4]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for steb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine",
)

C:\Users\pc\AppData\Local\Temp\ipykernel_29608\2459235318.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator


### Training


In [5]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define training arguments
args = SentenceTransformerTrainingArguments(
    output_dir='output/base_embedding_model',
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100
)


C:\Users\pc\AppData\Local\Temp\ipykernel_29608\1133874191.py:1: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingArguments


- num_train_epochs: The number of training rounds. We keep this at 1 for faster training but it is generally advised to increase this value.
- per_device_train_batch_size: The number of samples to process simultaneously on each device (e.g., GPU or CPU) during evaluation. Higher values generally means faster training.
- per_device_eval_batch_size: The number of samples to process simultaneously on each device (e.g., GPU or CPU) during evaluation. Higher values generally means faster evaluation.
- warmup_steps: The number of steps during which the learning rate will be linearly increased from zero to the initial learning rate defined for the training process. Note that we did not specify a custom learning rate for this training process.
- fp16: By enabling this parameter we allow for mixed precision training, where computations are performed using 16-bit floating-point numbers (FP16) instead of the default 32-bit (FP32). This reduces memory usage and potentially increases the training speed.

In [6]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

C:\Users\pc\AppData\Local\Temp\ipykernel_29608\2767052707.py:1: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import SentenceTransformerTrainer
Column 'hypothesis' is at index 1, whereas a column with this name is usually expected at index 0. Note that the column order can be important for some losses, e.g. MultipleNegativesRankingLoss will always consider the first column as the anchor and the second as the positive, regardless of the dataset column names. Consider renaming the columns to match the expected order, e.g.:
dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.069216
200,0.951879
300,0.892270
400,0.850261
500,0.822246
600,0.831090
700,0.807341
800,0.795445
900,0.771121
1000,0.762719


Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.70s/it]


TrainOutput(global_step=1563, training_loss=0.8135789418479836, metrics={'train_runtime': 420.7317, 'train_samples_per_second': 118.841, 'train_steps_per_second': 3.715, 'total_flos': 0.0, 'train_loss': 0.8135789418479836, 'epoch': 1.0})

 After training our model, we can use the evaluator to get the perfromance on this single tsk:
 

In [7]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.531152024044593, 'spearman_cosine': 0.6134434108883291}

- We get several different distance measures. The one we are interested in most is 'pearson_cosine', which is the cosine similarity between centered vectors. 
- It is a value between 0 and 1 where a higher value indicates higher degrees of similarity. We get a value of 0.53, which we consider a baseline throughout this chapter.


### Tip
Larger batch sizes tend to work better with multiple negative rankings (MNR) loss as a larger batch makes the task more difficult. The reason for this is that the model needs to find the best matching sentence from a larger set of potential pairs of sentences. You can adapt the code to try out different batch sizes and get a feeling of its effects.

## In-Depth Evaluation
- A good embedding model is more than just a good score on the STSB benchmark!
- The GLUE benchmark has a number of tasks for which we can evaluate our embedding model.
- However, there exist many more benchmarks that allow for the evaluation of embedding models
  - To unify this evaluation procedure, the massive Text Embedding Benchmark (MTEB) was developed
  - MTEB spans 8 embedding tasks that cover 58 datasets and 112 languages.

- Link: https://huggingface.co/spaces/mteb/leaderboard
- Link: https://pypi.org/project/mteb/

In [9]:
import mteb

# Choose evaluation task
# Choose evaluation task
evaluation = mteb.get_tasks(tasks=["Banking77Classification.v2"])

# Calculate results
results = mteb.evaluate(embedding_model, tasks=evaluation)
results

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\mteb\models\model_meta.py:753: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\mteb\models\model_meta.py:715: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),
Evaluating task Banking77Classification.v2:   0%|          | 0/1 [00:00<?, ?it/s]d:\2026-courses\LLMs-Handson\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\datasets--mteb--banking77. Caching files will still work but in a degraded version that might require more space on your disk. This warning can

ModelResult(model_name='SentenceTransformer based on google-bert/bert-base-uncased', model_revision='86b5e0934494bd15c9632b12f734a8a67f723594', task_results=[...](#1), ...)

In [13]:
results

ModelResult(model_name='SentenceTransformer based on google-bert/bert-base-uncased', model_revision='86b5e0934494bd15c9632b12f734a8a67f723594', task_results=[...](#1), ...)

In [28]:
# Inspect the full results structure
print("Model Name:", results.model_name)
print("\nModel Revision:", results.model_revision)
print("\nTask Results:")
for task_result in results.task_results:
    print(f"\n  Task: {task_result.task_name}")
    print(f"  Scores: {task_result.scores['test'][0]['scores_per_experiment']}")
    if hasattr(task_result, 'evaluation_time'):
        print(f"  Evaluation Time: {task_result.evaluation_time}")
    if hasattr(task_result, 'main_score'):
        print(f"  Main Score: {task_result.main_score}")

Model Name: SentenceTransformer based on google-bert/bert-base-uncased

Model Revision: 86b5e0934494bd15c9632b12f734a8a67f723594

Task Results:

  Task: Banking77Classification.v2
  Scores: [{'accuracy': 0.5744473342002601, 'f1': 0.5718957161277722, 'f1_weighted': 0.5718924851449991, 'precision': 0.5822662270049382, 'precision_weighted': 0.5822825784119708, 'recall': 0.5744588744588744, 'recall_weighted': 0.5744473342002601, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.6007802340702211, 'f1': 0.5994701248522412, 'f1_weighted': 0.599596883805227, 'precision': 0.613377705305322, 'precision_weighted': 0.6135057675394631, 'recall': 0.6006410256410256, 'recall_weighted': 0.6007802340702211, 'ap': None, 'ap_weighted': None}, {'accuracy': 0.5842002600780234, 'f1': 0.5809730978162526, 'f1_weighted': 0.5811132039824065, 'precision': 0.5936165854030808, 'precision_weighted': 0.5937433249859798, 'recall': 0.5840659340659341, 'recall_weighted': 0.5842002600780234, 'ap': None, 'ap_weighted': No

### Tip

Whenever you are done training and evaluating your model, it is important to restart the notebook. This will clear your VRAM up for the next training examples throughout this chapter. By restarting the notebook, we can be sure that all VRAM is cleared.

 ## VRAM Clean-up - You will need to run the code below to partially empty the VRAM (GPU RAM). If that does not work, it is advised to restart the notebook instead. You can check the resources on the right-hand side (if you are using Google Colab) to check whether the used VRAM is indeed low. You can also run !nivia-smi to check current usage.

In [ ]:
# # Empty and delete trainer/model
# trainer.accelerator.clear()
# del trainer, embedding_model

# # Garbage collection and empty cache
# import gc
# import torch

# gc.collect()
# torch.cuda.empty_cache()



In [29]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## Loss Functions
- We trained our model using softmax loss to illustrate how one of the first sentence-transformers models was trained.
- However, not only is there a large variety of loss functions to choose from, but softmax loss is generally not advised as there are more: https://www.sbert.net/docs/package_reference/sentence_transformer/losses.html

- Instead of going through every single loss function out there, there are two loss functions that are typically used and seem to perform generally well, namely:
 - Cosine similarity
 - Multiple negatives ranking (MNR) loss

### Note
There are many more loss functions to choose from than just those discussed here. For example, a loss like MarginMSE works great for training or fine-tuning a cross-encoder. There are a number of interesting loss functions: https://www.sbert.net/docs/package_reference/sentence_transformer/losses.html

### Cosine Similary loss
- The cosine similarity loss is an intuitive and easy-to-use loss that works across many different use cases and datasets. It is typically used in semantic textual similarity tasks. 
- In these tasks, a similarity score is assigned to the pairs of texts over which we optimize the model.
- Instead of having strictly positive or negative pairs of sentences, we assume pairs of sentences that are similar or dissimilar to a certain degree.
- Typically, this value lies between 0 and 1 to indicate dissimilarity and similarity, respectively 
- Cosine similarity loss is straightforward—it calculates the cosine similarity between the two embeddings of the two texts and compares that to the labeled similarity score. The model will learn to recognize the degree of similarity between sentences.

Cosine similarity loss intuitively works best using data where you have pairs of sentences and labels that indicate their similarity between 0 and 1. To use this loss with our NLI dataset, we need to convert the entailment (0), neutral (1), and contradiction (2) labels to values between 0 and 1. The entailment represents a high similarity between the sentences, so we give it a similarity score of 1. In contrast, since both neutral and contradiction represent dissimilarity, we give these labels a similarity score of 0:

In [30]:
from datasets import load_dataset, Dataset

# Load MNLI dataset from GLUE
## 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")
# (neutral/contradiction)=0 and (entailment)=1
mapping = {2: 0, 1: 0, 0:1}
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})


In [31]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="cosineloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2426.65it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,0.230803
200,0.172327
300,0.170321
400,0.161277
500,0.152638
600,0.158865
700,0.149555
800,0.155277
900,0.150228
1000,0.147262


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.21s/it]


TrainOutput(global_step=1563, training_loss=0.15734979882121314, metrics={'train_runtime': 310.3827, 'train_samples_per_second': 161.091, 'train_steps_per_second': 5.036, 'total_flos': 0.0, 'train_loss': 0.15734979882121314, 'epoch': 1.0})

In [32]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.7243138926457293, 'spearman_cosine': 0.7269502596832503}

As we can see above, A pearson cosine similarity is 0.72 is a big improvement compared to the softmax loss example, which scored 0.53. This demonstrates the impact the loss function can have on performance

### VRAM Clean-up

In [33]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

### Multiple Negatives Ranking Loss
- Multiple negatives ranking (MNR) loss,6 often referred to as **InfoNCE** or **NTXentLoss** is a loss that uses either positive pairs of sentences or triplets that contain a pair of positive sentences and an additional unrelated sentence. This unrelated sentence is called a negative and represents the dissimilarity between the positive sentences.

- For example, you might have pairs of question/answer, image/image caption, paper title/paper abstract, etc.
- The great thing about these pairs is that we can be confident they are hard positive pairs. 
- In MNR loss (img10), negative pairs are constructed by mixing a positive pair with another positive pair. In the example of a paper title and abstract, you would generate a negative pair by combining the title of a paper with a completely different abstract. These negatives are called in-batch negatives and can also be used to generate the triplets.
- After having generated these positive and negative pairs, we calculate their embeddings and apply cosine similarity.
- These similarity scores are then used to answer the question, are these pairs negative or positive? In other words, it is treated as a classification task and we can use cross-entropy loss to optimize the model.
- To make these triplets we start with an anchor sentence (i.e., labeled as the “premise”), which is used to compare other sentences. Then, using the MNLI dataset, we only select sentence pairs that are positive (i.e., labeled as “entailment”). To add negative sentences, we randomly sample sentences as the “hypothesis.”

In [ ]:
import random
from tqdm import tqdm
from datasets import Dataset, load_dataset

# # Load MNLI dataset from GLUE
mnli = load_dataset("glue", "mnli", split="train").select(range(50_000))
mnli = mnli.remove_columns("idx")
mnli = mnli.filter(lambda x: True if x['label'] == 0 else False)

# Prepare data and add a soft negative
train_dataset = {"anchor": [], "positive": [], "negative": []}
soft_negatives = list(mnli["hypothesis"])
random.shuffle(soft_negatives)
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
    train_dataset["anchor"].append(row["premise"])
    train_dataset["positive"].append(row["hypothesis"])
    train_dataset["negative"].append(soft_negative)
train_dataset = Dataset.from_dict(train_dataset)
len(train_dataset)


16875it [00:01, 12281.13it/s]


16875

In [39]:
# Define the evaluator
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine" )

In [40]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3754.69it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,0.314707
200,0.100829
300,0.077668
400,0.064334
500,0.070635


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.99s/it]


TrainOutput(global_step=528, training_loss=0.12234853659615372, metrics={'train_runtime': 154.8888, 'train_samples_per_second': 108.949, 'train_steps_per_second': 3.409, 'total_flos': 0.0, 'train_loss': 0.12234853659615372, 'epoch': 1.0})

In [41]:
# Evaluate our trained model
evaluator(embedding_model)



{'pearson_cosine': 0.8098104325025374, 'spearman_cosine': 0.8121727941393816}

Compared to our previously trained model with softmax loss (0.72), our model with MNR loss (0.80) seems to be much more accurate!

### Tip
Larger batch sizes tend to be better with MNR loss as a larger batch makes the task more difficult. The reason for this is that the model needs to find the best matching sentence from a larger set of potential pairs of sentences. You can adapt the code to try out different batch sizes and get a feeling of the effects.

### Downside of the above approach
There is a downside to how we used this loss function. Since negatives are sampled from other question/answer pairs, these in-batch or “easy” negatives that we used could potentially be completely unrelated to the question. As a result, the embedding model’s task of then finding the right answer to a question becomes quite easy. Instead, we would like to have negatives that are very related to the question but not the right answer. These negatives are called **hard negatives**. Since this would make the task more difficult for the embedding model as it has to learn more nuanced representations, the embedding model’s performance generally improves quite a bit.

- A good example of a hard negative is the following. Let’s assume we have the following question: “How many people live in Amsterdam?” A related answer to this question would be: “Almost a million people live in Amsterdam.” 
- To generate a good hard negative, we ideally want the answer to contain something about Amsterdam and the number of people living in this city. 
  - For example: “More than a million people live in Utrecht, which is more than Amsterdam.” This answer relates to the question but is not the actual answer, so this would be a good hard negative. img11 illustrates the differences between easy and hard negatives.

Gathering negatives can roughly be divided into the following three processes:
 - Easy negatives: Through randomly sampling documents as we did before.
 - Semi-hard negatives: Using a pretrained embedding model, we can apply cosine similarity on all sentence embeddings to find those that are highly related. Generally, this does not lead to hard negatives since this method merely finds similar sentences, not question/answer pairs.
 - Hard negatives: These often need to be either manually labeled (for instance, by generating semi-hard negatives) or you can use a generative model to either judge or generate sentence pairs.

## Understanding the `mnrloss_embedding_model` Directory

This directory contains your **trained embedding model** using **Multiple Negatives Ranking (MNR) Loss**. 

### 📁 Directory Structure Overview

The directory contains **checkpoints** - snapshots of your model saved at different training steps:
- `checkpoint-500/` - Model saved at step 500
- `checkpoint-528/` - Model saved at step 528 (final checkpoint, end of training)

---

### 📄 Key Files Explained

#### 1. **Model Files**
- **`model.safetensors`** (440 MB)
  - The actual trained neural network weights
  - SafeTensors format is faster and safer than PyTorch's .pt format
  - Contains all the BERT parameters fine-tuned for your embedding task
  
#### 2. **Configuration Files**
- **`config.json`**
  - Model architecture configuration
  - Shows: 768 hidden dimensions, 12 layers, 12 attention heads
  - BERT-base architecture specifications
  
- **`modules.json`**
  - Defines the sentence-transformer pipeline structure
  - Module 0: Transformer (BERT)
  - Module 1: Pooling layer (mean pooling)
  
- **`config_sentence_transformers.json`**
  - Sentence-transformers specific configuration
  
- **`sentence_bert_config.json`**
  - Additional SBERT settings

#### 3. **Tokenizer Files**
- **`tokenizer.json`** (466 KB)
  - Fast tokenizer configuration
  - Contains vocabulary and tokenization rules
  
- **`tokenizer_config.json`**
  - Tokenizer settings (lowercase, special tokens, etc.)

#### 4. **Training State Files**
- **`trainer_state.json`**
  - Records training progress and metrics
  - Shows loss values at each logging step
  - Training took 528 steps total (1 epoch)
  
- **`training_args.bin`**
  - Saved training arguments
  - Batch size, learning rate, warmup steps, etc.

#### 5. **Optimizer & Training Files**
- **`optimizer.pt`** (335 MB)
  - Optimizer (AdamW) state
  - Needed to resume training from this checkpoint
  
- **`scheduler.pt`**
  - Learning rate scheduler state
  
- **`scaler.pt`**
  - Mixed precision (FP16) scaler state
  - Helps prevent numerical instability during FP16 training
  
- **`rng_state.pth`**
  - Random number generator state
  - Ensures reproducibility if you resume training

#### 6. **Pooling Configuration**
- **`1_Pooling/config.json`**
  - Pooling layer configuration
  - Uses **mean pooling** to convert token embeddings to sentence embeddings
  - 768-dimensional output

#### 7. **Documentation**
- **`README.md`**
  - Auto-generated model card
  - Documents: model details, training data, hyperparameters, usage examples
  - Ready to upload to Hugging Face Hub!

---

### 📊 What This Model Does

**Model Type:** Sentence Embedding Model (based on BERT-base-uncased)
- **Input:** Text sentences/paragraphs
- **Output:** 768-dimensional dense vectors
- **Training Method:** Multiple Negatives Ranking Loss with anchor-positive-negative triplets
- **Training Data:** 16,875 samples from MNLI dataset

**Loss Function:** MultipleNegativesRankingLoss (MNR)
- Uses triplets: anchor, positive (similar), negative (dissimilar)
- Optimizes so similar sentences are close in vector space
- Better than SoftmaxLoss for retrieval and semantic similarity tasks

---

### 🎯 Key Training Information

From `trainer_state.json`:

| Step | Epoch | Loss    | Learning Rate |
|------|-------|---------|---------------|
| 100  | 0.19  | 0.3147  | 4.95e-05      |
| 200  | 0.38  | 0.1008  | 3.84e-05      |
| 300  | 0.57  | 0.0777  | 2.68e-05      |
| 400  | 0.76  | 0.0643  | 1.51e-05      |
| 500  | 0.95  | 0.0706  | 3.39e-06      |
| 528  | 1.00  | -       | Final         |

**Training Time:** ~2.5 minutes
**Training Setup:** 32 batch size, FP16 precision, 100 warmup steps

---

### 💾 File Sizes & What You Need

**Essential files to use the model (~440 MB):**
- `model.safetensors`
- `config.json`
- `tokenizer.json`
- `tokenizer_config.json`
- `modules.json`
- `1_Pooling/config.json`

**Only needed to resume training (~335 MB):**
- `optimizer.pt`
- `scheduler.pt`
- `scaler.pt`
- `rng_state.pth`
- `training_args.bin`

---

### 🚀 How to Use This Model

```python
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer('notebooks/mnrloss_embedding_model/checkpoint-528')

# Generate embeddings
sentences = [
    "I love machine learning",
    "Deep learning is fascinating",
    "I enjoy cooking pasta"
]
embeddings = model.encode(sentences)

# Calculate similarity
similarities = model.similarity(embeddings, embeddings)
print(similarities)
```

---

### ✅ Summary

The `mnrloss_embedding_model` directory contains:
1. ✅ A fully trained sentence embedding model
2. ✅ Based on BERT-base-uncased
3. ✅ Fine-tuned with MNR Loss on MNLI data
4. ✅ Produces 768-dimensional embeddings
5. ✅ Ready to use for semantic similarity and retrieval tasks
6. ✅ Can be uploaded to Hugging Face Hub (README included)

In [ ]:
# Quick inspection of the mnrloss_embedding_model directory
import os
import json

checkpoint_dir = "mnrloss_embedding_model/checkpoint-528"

print("=" * 80)
print("📂 MNR LOSS EMBEDDING MODEL - FILE STRUCTURE")
print("=" * 80)

# Check if directory exists
if os.path.exists(checkpoint_dir):
    files = []
    for root, dirs, filenames in os.walk(checkpoint_dir):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            size = os.path.getsize(filepath)
            relative_path = os.path.relpath(filepath, checkpoint_dir)
            files.append((relative_path, size))
    
    # Sort by size (largest first)
    files.sort(key=lambda x: x[1], reverse=True)
    
    # Display files with sizes
    print(f"\n📁 Directory: {checkpoint_dir}\n")
    print(f"{'File Name':<50} {'Size':>15}")
    print("-" * 65)
    
    total_size = 0
    for filename, size in files:
        total_size += size
        if size > 1024**3:  # GB
            size_str = f"{size / 1024**3:.2f} GB"
        elif size > 1024**2:  # MB
            size_str = f"{size / 1024**2:.2f} MB"
        elif size > 1024:  # KB
            size_str = f"{size / 1024:.2f} KB"
        else:
            size_str = f"{size} bytes"
        
        print(f"{filename:<50} {size_str:>15}")
    
    print("-" * 65)
    if total_size > 1024**3:
        total_str = f"{total_size / 1024**3:.2f} GB"
    else:
        total_str = f"{total_size / 1024**2:.2f} MB"
    print(f"{'TOTAL':<50} {total_str:>15}")
    
    # Read and display key information
    print("\n" + "=" * 80)
    print("🔍 KEY MODEL INFORMATION")
    print("=" * 80)
    
    # Read config
    with open(os.path.join(checkpoint_dir, "config.json")) as f:
        config = json.load(f)
    
    print(f"\n📐 Model Architecture:")
    print(f"  • Type: {config['model_type'].upper()}")
    print(f"  • Hidden Size: {config['hidden_size']}")
    print(f"  • Number of Layers: {config['num_hidden_layers']}")
    print(f"  • Attention Heads: {config['num_attention_heads']}")
    print(f"  • Vocabulary Size: {config['vocab_size']:,}")
    print(f"  • Max Sequence Length: {config['max_position_embeddings']}")
    
    # Read pooling config
    with open(os.path.join(checkpoint_dir, "1_Pooling", "config.json")) as f:
        pooling = json.load(f)
    
    print(f"\n🎯 Pooling Configuration:")
    print(f"  • Embedding Dimension: {pooling['embedding_dimension']}")
    print(f"  • Pooling Mode: {pooling['pooling_mode']}")
    
    # Read trainer state
    with open(os.path.join(checkpoint_dir, "trainer_state.json")) as f:
        trainer = json.load(f)
    
    print(f"\n📊 Training Statistics:")
    print(f"  • Total Steps: {trainer['global_step']}")
    print(f"  • Epochs: {trainer['epoch']}")
    print(f"  • Batch Size: {trainer['train_batch_size']}")
    
    if trainer['log_history']:
        final_loss = trainer['log_history'][-1]['loss']
        print(f"  • Final Loss: {final_loss:.4f}")
    
    print("\n" + "=" * 80)
else:
    print(f"❌ Directory not found: {checkpoint_dir}")
    print("Make sure you've trained the model first!")

In [43]:
# Load and test the trained MNR Loss embedding model
from sentence_transformers import SentenceTransformer
import numpy as np

print("=" * 80)
print("LOADING AND TESTING MNR LOSS EMBEDDING MODEL")
print("=" * 80)

# Load the model
model_path = "mnrloss_embedding_model/checkpoint-528"
print(f"\n Loading model from: {model_path}")

try:
    model = SentenceTransformer(model_path)
    print("Model loaded successfully!")
    
    # Test sentences
    sentences = [
        "The cat sits on the mat",
        "A feline rests on the carpet",  # Similar to sentence 1
        "Machine learning is fascinating",  # Different topic
        "Deep learning and AI are interesting",  # Similar to sentence 3
        "I love pizza and pasta"  # Different topic
    ]
    
    print(f"\n Encoding {len(sentences)} test sentences...")
    embeddings = model.encode(sentences, convert_to_tensor=False)
    
    print(f" Embeddings generated!")
    print(f"   Shape: {embeddings.shape}")
    print(f"   Dimension: {embeddings.shape[1]}")
    
    # Calculate similarity matrix
    print(f"\n Calculating similarity matrix...")
    from sklearn.metrics.pairwise import cosine_similarity
    
    similarity_matrix = cosine_similarity(embeddings)
    
    print(f"\n SIMILARITY MATRIX (Cosine Similarity)")
    print("   Higher values = more similar sentences\n")
    
    # Print header
    print(f"{'':>5}", end="")
    for i in range(len(sentences)):
        print(f"  S{i+1}  ", end="")
    print()
    print("   " + "-" * (7 * len(sentences)))
    
    # Print similarity matrix
    for i, row in enumerate(similarity_matrix):
        print(f"S{i+1} |", end="")
        for val in row:
            print(f" {val:5.2f}", end="")
        print(f"  | {sentences[i][:40]}...")
    
    print("\n Test Sentences:")
    for i, sent in enumerate(sentences):
        print(f"   S{i+1}: {sent}")
    
    print("\n Observations:")
    print(f"   • S1 ↔ S2: {similarity_matrix[0][1]:.3f} (cat/feline - SIMILAR ✓)")
    print(f"   • S1 ↔ S3: {similarity_matrix[0][2]:.3f} (cat/ML - DIFFERENT ✓)")
    print(f"   • S3 ↔ S4: {similarity_matrix[2][3]:.3f} (ML/AI - SIMILAR ✓)")
    
    print("\n" + "=" * 80)
    
except Exception as e:
    print(f" Error: {e}")
    print("Make sure the model directory exists and contains all required files!")

LOADING AND TESTING MNR LOSS EMBEDDING MODEL

 Loading model from: mnrloss_embedding_model/checkpoint-528


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 496.26it/s]


Model loaded successfully!

 Encoding 5 test sentences...
 Embeddings generated!
   Shape: (5, 768)
   Dimension: 768

 Calculating similarity matrix...

 SIMILARITY MATRIX (Cosine Similarity)
   Higher values = more similar sentences

       S1    S2    S3    S4    S5  
   -----------------------------------
S1 |  1.00  0.74  0.17  0.07  0.05  | The cat sits on the mat...
S2 |  0.74  1.00  0.17  0.15  0.14  | A feline rests on the carpet...
S3 |  0.17  0.17  1.00  0.74  0.13  | Machine learning is fascinating...
S4 |  0.07  0.15  0.74  1.00  0.16  | Deep learning and AI are interesting...
S5 |  0.05  0.14  0.13  0.16  1.00  | I love pizza and pasta...

 Test Sentences:
   S1: The cat sits on the mat
   S2: A feline rests on the carpet
   S3: Machine learning is fascinating
   S4: Deep learning and AI are interesting
   S5: I love pizza and pasta

 Observations:
   • S1 ↔ S2: 0.742 (cat/feline - SIMILAR ✓)
   • S1 ↔ S3: 0.168 (cat/ML - DIFFERENT ✓)
   • S3 ↔ S4: 0.736 (ML/AI - SIMILAR 

In [45]:
# Compare checkpoint-500 vs checkpoint-528 (final)
import json
import os

print("=" * 80)
print("COMPARING CHECKPOINTS: 500 vs 528")
print("=" * 80)

checkpoints = ["mnrloss_embedding_model/checkpoint-500", "mnrloss_embedding_model/checkpoint-528"]

for cp_path in checkpoints:
    if os.path.exists(cp_path):
        cp_name = cp_path.split("/")[-1]
        print(f"\n {cp_name.upper()}")
        print("-" * 40)
        
        # Read trainer state
        with open(os.path.join(cp_path, "trainer_state.json")) as f:
            trainer = json.load(f)
        
        print(f"  Global Step: {trainer['global_step']}")
        print(f"  Epoch: {trainer['epoch']:.2f}")
        
        # Get last logged loss
        if trainer['log_history']:
            last_log = trainer['log_history'][-1]
            print(f"  Last Logged Loss: {last_log['loss']:.4f}")
            print(f"  Learning Rate: {last_log['learning_rate']:.2e}")

print("\n" + "=" * 80)
print("RECOMMENDATION")
print("=" * 80)
print("\nUse checkpoint-528 - it's the FINAL checkpoint after completing 1 epoch")
print("   • More training steps (528 vs 500)")
print("   • Completed full training cycle")
print("   • Most optimized version of your model")
print("\n checkpoint-500 is just an intermediate save (every 500 steps)")
print("\n" + "=" * 80)

COMPARING CHECKPOINTS: 500 vs 528

 CHECKPOINT-500
----------------------------------------
  Global Step: 500
  Epoch: 0.95
  Last Logged Loss: 0.0706
  Learning Rate: 3.39e-06

 CHECKPOINT-528
----------------------------------------
  Global Step: 528
  Epoch: 1.00
  Last Logged Loss: 0.0706
  Learning Rate: 3.39e-06

RECOMMENDATION

Use checkpoint-528 - it's the FINAL checkpoint after completing 1 epoch
   • More training steps (528 vs 500)
   • Completed full training cycle
   • Most optimized version of your model

 checkpoint-500 is just an intermediate save (every 500 steps)



## Fine-Tuning an Embedding Model
- In previuos section, we went through the basics of training an embedding model from scratch and saw how we could leverage loss functions to further optimize the performance
- This approach, although quite powerful, requires creating an embedding model from scratch. This process can be quite costly and time consuming.
- Instead, the sentence-transformers framework allows nearly all embedding models to be used as a base fine-tuning. We can choose an embedding model that was already trained on a large amount of data and fine-tune it for our specifc use case.

## Supervised Fine-tuning (SFT)

- The most straightforward way to fine-tune an embedding model is to repeat the process of training our model as we did before but replace the 'bert-base-uncased' with a pretrained sentence-transformers model.
- There are many to choose from but generally, all-MiniLM-L6-v2 performs well across many use cases and due to its small size is quite fast.


## VRAM Clean up

In [1]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [4]:
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)


In [5]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

C:\Users\pc\AppData\Local\Temp\ipykernel_39992\3640762123.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses, SentenceTransformer
C:\Users\pc\AppData\Local\Temp\ipykernel_39992\3640762123.py:2: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import SentenceTransformerTrainer
C:\Users\pc\AppData\Local\Temp\ipykernel_39992\3640762123.py:3: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import SentenceTransformerTrainingA

Step,Training Loss
100,0.155216
200,0.113601
300,0.119578
400,0.112722
500,0.111110
600,0.099545
700,0.117749
800,0.102287
900,0.101068
1000,0.100672


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.24it/s]


TrainOutput(global_step=1563, training_loss=0.1093088239717392, metrics={'train_runtime': 131.4838, 'train_samples_per_second': 380.275, 'train_steps_per_second': 11.887, 'total_flos': 0.0, 'train_loss': 0.1093088239717392, 'epoch': 1.0})

In [6]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.8501675973104518, 'spearman_cosine': 0.8492356946075419}

Although a score of 0.85 is the highest we have seen thus far, the pretrained model that we used for fine-tuning was already trained on the full MNLI dataset, whereas we only used 50,000 examples. It might seem redundant but this example demonstrates how to fine-tune a pretrained embedding model on your own data.

### Tip
- domain model like 'all-mpnet-base-v2', you can also perform masked language modeling on the pretrained BERT model to first adapt it to your domain. Then, you can use this fine-tuned BERT model as the base for training your embedding model. This is a form of domain adaptation. In the next chapter, we will apply masked language modeling on a pretrained model.

- Note that the main difficulty of training or fine-tuning your model is finding the right data. With these models, we not only want to have very large datasets, but the data in itself needs to be of high quality. Developing positive pairs is generally straightforward but adding hard negative pairs significantly increases the difficulty of creating quality data.

In [12]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## Augment SBERT
- A disadvantage of training or fine-tuning these embedding models is that they often require substantial training data. 
- Many of these models are trained with more than a billion sentence pairs. 
- Extracting such a high number of sentence pairs for your use case is generally not possible as in many cases, there are only a couple of thousand labeled data points available.
- Fortunately, there is a way to augment your data such that an embedding model can be fine-tuned when there is only a little labeled data available.
- This procedure is referred to as Augmented SBERT.
- Link: https://arxiv.org/abs/2010.08240
- SBERT goal: Augment the small amount of labeled data such that they can be used for regular training. 
- SBERT makes use of the slow and more accurate cross-encoder architecture (BERT) to augment and label a larger set of input pairs. These newly labeled pairs are then used for fine-tuning a bi-encoder (SBERT).
- As shown in img12, Augmented SBERT involves the following steps:
  - Fine-tune a cross-encoder (BERT) using a small, annotated dataset (gold dataset)
  - Create new sentence pairs
  - Label new sentence pairs with the fine-tuned cross-encoder (silver dataset)
  - Train a bi-encoder (SBERT) on the extended dataset (gold + silver dataset).

Here, a gold dataset is a small but fully annotated dataset that holds the ground truth. A silver dataset is also fully annotated but is not necessarily the ground truth as it was generated through predictions of the cross-encoder.

Before we get into the preceding steps, let’s first prepare the data. Instead of our original 50,000 documents, we take a subset of 10,000 documents to simulate a setting where we have limited annotated data. As we did in our example with cosine similarity loss, give entailment a score of 1 whereas neutral and contradiction get a score of 0:

### Step1: Fine-tune a cross-encoder


In [15]:
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader 

In [16]:
# Prepare a small set of 10000 documents for the cross-encoder
dataset = load_dataset("glue", "mnli", split="train").select(range(10_000))
mapping = {2: 0, 1: 0, 0:1}

# Data Loader
gold_examples = [
    InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
    for row in tqdm(dataset)
]
gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)

# Pandas DataFrame for easier data handling
gold = pd.DataFrame(
    {
    'sentence1': dataset['premise'],
    'sentence2': dataset['hypothesis'],
    'label': [mapping[label] for label in dataset['label']]
    }
)

100%|██████████| 10000/10000 [00:00<00:00, 20201.98it/s]


In [17]:
from sentence_transformers.cross_encoder import CrossEncoder

# Train a cross encoder on the gold dataset
cross_encoder = CrossEncoder('bert-base-uncased', num_labels=2)
# Train a cross-encoder on the gold dataset
cross_encoder.fit(
    train_dataloader=gold_dataloader,
    epochs=1,
    show_progress_bar=True,
    warmup_steps=100,
    use_amp=False
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3754.79it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

Step,Training Loss


### step2: Create new sentence pairs


In [ ]:
# Prepare the silver dataset by predicting labels with the cross-encoder
silver = load_dataset("glue", "mnli", split="train").select(range(10_000,50_000))
pairs = list(zip(silver['premise'],silver['hypothesis']))

## step3: Label new sentence pairs with the fine-tuned cross-encoder (silver dataset)


In [19]:
import numpy as np

# Label the sentence pairs using our fine-tuned cross-encoder
output = cross_encoder.predict(pairs, apply_softmax=True, show_progress_bar=True)
silver = pd.DataFrame(
    {
        "sentence1": silver["premise"],
        "sentence2": silver["hypothesis"],
        "label": np.argmax(output, axis=1)
    }
)

Batches: 100%|██████████| 1250/1250 [01:03<00:00, 19.73it/s]


### Step4:Train a bi-encoder (SBERT) on the extended dataset (gold + silver dataset) 

In [20]:
# Combine gold + silver
data = pd.concat([gold, silver], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

In [21]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [22]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="augmented_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2519.32it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,0.214235
200,0.157267
300,0.142744
400,0.142682
500,0.138999
600,0.134024
700,0.131224
800,0.129173
900,0.132197
1000,0.128873


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.08s/it]


TrainOutput(global_step=1563, training_loss=0.13884991861198168, metrics={'train_runtime': 358.7079, 'train_samples_per_second': 139.384, 'train_steps_per_second': 4.357, 'total_flos': 0.0, 'train_loss': 0.13884991861198168, 'epoch': 1.0})

In [23]:
# Evaluate our trained model
evaluator(embedding_model)



{'pearson_cosine': 0.6698869877389844, 'spearman_cosine': 0.6880285353140563}

In [24]:
trainer.accelerator.clear()

[]

### Step 5: Evaluate without silver dataset


In [ ]:
# Only Gold data
data = pd.concat([gold], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="gold_only_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3158.68it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,0.225501
200,0.171618
300,0.161116


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.59s/it]


TrainOutput(global_step=313, training_loss=0.18540507078932497, metrics={'train_runtime': 69.3816, 'train_samples_per_second': 144.13, 'train_steps_per_second': 4.511, 'total_flos': 0.0, 'train_loss': 0.18540507078932497, 'epoch': 1.0})

In [26]:
evaluator(embedding_model)

{'pearson_cosine': 0.67836540701911, 'spearman_cosine': 0.6911698559872531}

### how we can test the quality of the silver data
This method allows us to increase the size of datasets that you already have available without the need to manually label hundreds of thousands of sentence pairs. You can test the quality of your silver data by also training your embedding model only on the gold dataset. The difference in performance indicates how much your silver dataset potentially adds to the quality of the model.

## Works with the Complete Cleanup Pattern
Each step serves a purpose:

- Step 1 (accelerator.clear()): Releases Accelerate-managed resources
- Step 2 (del): Removes Python references
- Step 3 (gc.collect()): Forces Python garbage collector to reclaim memory
- Step 4 (torch.cuda.empty_cache()): Returns cached GPU memory to the system

When to Use It: Use this cleanup pattern:

- Between training different models in the same notebook
- After evaluation when you won't use the model anymore
- Before loading a new large model
- When you're running low on VRAM

Note: As mentioned in your notebook, if this partial cleanup doesn't free enough me

In [33]:
# Step 1: Clear accelerator-managed resources
trainer.accelerator.clear()

# Step 2: Delete Python objects
del trainer, embedding_model

# Step 3: Force garbage collection
import gc
gc.collect()

# Step 4: Empty CUDA cache
import torch
torch.cuda.empty_cache()

## Unsupervised Learning
- To build an embedding model, we need typically labeled data.
- Not all real-world datasets come with a nice set of labels that we can use.
- Instead, we look for techniques to train the model without any predetermined labels: Unsupervised Learning
- Many approaches exist:
  - Simple Contrastive Learning of Sentence Embeddings (SimCSE): https://arxiv.org/abs/2104.08821
  - Contrastive Tension (CT): https://www.diva-portal.org/smash/record.jsf?amp%3Bdswid=-528&pid=diva2%3A1684806&dswid=3532
  -  Transformer-based Sequential Denoising Auto-Encoder (TSDAE): https://arxiv.org/abs/2104.06979
  - Generative Pseudo-Labeling (GPL): https://arxiv.org/abs/2112.07577

 ### Transformer-Based Sequential Denoising Auto-Encoder
- TSDAE is a very elegant approach to creating an embedding model with unsupervised learning.
- The method assumes that we have no labeled data at all and does not require us to artificially create labels.
- The underlying idea of TSDAE is that we add noise to the input sentence by removing a certain percentage of words from it.
- This “damaged” sentence is put through an encoder, with a pooling layer on top of it, to map it to a sentence embedding.
- From this sentence embedding, a decoder tries to reconstruct the original sentence from the “damaged” sentence but without the artificial noise.
- The main concept here is that the more accurate the sentence embedding is, the more accurate the reconstructed sentence will be.
- This method is very similar to masked language modeling, where we try to reconstruct and learn certain masked words. Here, instead of reconstructing masked words, we try to reconstruct the entire sentence.
- After training, we can use the encoder to generate embeddings from text since the decoder is only used for judging whether the embeddings can accurately reconstruct the original sentence


Since we only need a bunch of sentences without any labels, training this model is straightforward. We start by downloading an external tokenizer, which is used for the denoising procedure:



